# Exemplos do Capítulo 4: A Matemática dos Sistemas Biológicos

Este notebook contém os exemplos práticos de código discutidos no **Capítulo 4** do curso de **Bioinformática para Biologia de Sistemas**.

### Conteúdo do Notebook:
1. **Modelos Discretos e Recursivos**: Simulação do Mapa Logístico (Determinista vs. Estocástico com Ruído Multiplicativo)
2. **Resolução de EDOs com SciPy**: Integração numérica de sistemas acoplados de primeira ordem
3. **Modelo Lotka-Volterra (Predador-Presa)**: Simulação e visualização de trajetórias e retrato de fase
4. **Análise de Estabilidade Linear**: Linearização simbólica, cálculo da matriz Jacobiana com `SymPy` e autovalores com `SciPy`
5. **Teoria de Bifurcações**: Varredura paramétrica de equilíbrios e geração do Diagrama de Bifurcação Pitchfork Supercrítica
6. **Oscilações e Ciclos Limite**: Simulação e retrato de fase do Oscilador de Van der Pol
7. **Análise de Sensibilidade Paramétrica Local**: Cálculo de coeficientes de sensibilidade normalizados via perturbação numérica
8. **Análise de Sensibilidade Global (GSA)**: Método de Sobol para decomposição de variância paramétrica usando a biblioteca `SALib`

## Instalação de Dependências

Execute a célula abaixo para garantir que todas as bibliotecas necessárias estejam devidamente instaladas no seu ambiente Python.

In [ ]:
# Instalar as bibliotecas necessarias para o capitulo
!pip install numpy scipy matplotlib sympy SALib

## 1. Modelos Discretos e Recursivos (O Mapa Logístico)

O Mapa Logístico descreve a dinâmica de uma população sob restrição de recursos em tempo discreto:
$$x_{t+1} = r x_t (1 - x_t)$$

Abaixo, simulamos e comparamos o comportamento da população sob a forma **determinista** clássica contra a versão **estocástica** com ruído demográfico multiplicativo log-normal:
$$x_{t+1} = \text{clip}\left( r x_t (1 - x_t) \cdot e^{\eta_t}, 0.0, 1.0 \right)$$
onde $\eta_t \sim \mathcal{N}(0, \sigma^2)$ é uma flutuação aleatória gaussiana.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametros do Mapa Logistico
r = 3.8       # Comportamento caotico
sigma = 0.02  # Intensidade do ruido estocastico
steps = 50
x_det = np.zeros(steps)
x_est = np.zeros(steps)
x_det[0] = x_est[0] = 0.5

# Simulacao Recursiva
for t in range(steps - 1):
    # Determinista
    x_det[t+1] = r * x_det[t] * (1 - x_det[t])
    # Estocastico com ruido multiplicativo (truncado em 0 e 1)
    noise = np.random.normal(0, sigma)
    val = r * x_est[t] * (1 - x_est[t]) * np.exp(noise)
    x_est[t+1] = np.clip(val, 0.0, 1.0)

# Visualizar
plt.figure(figsize=(10, 4))
plt.plot(x_det, 'b-o', label='Determinista')
plt.plot(x_est, 'r--x', label='Estocastico')
plt.xlabel('Tempo (t)')
plt.ylabel('Populacao x(t)')
plt.legend()
plt.grid(True)
plt.title('Mapa Logistico: Determinista vs. Estocastico com Ruido Multiplicativo')
plt.show()

## 2. Resolvendo EDOs com SciPy

Muitos modelos contínuos em Biologia de Sistemas envolvem sistemas de equações diferenciais ordinárias (EDOs) acopladas. Abaixo, integramos numericamente um sistema genérico de duas variáveis acopladas usando a função `odeint` do `SciPy`:

$$\frac{dx}{dt} = \alpha x - \beta x y$$
$$\frac{dy}{dt} = -y + x y$$

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Definir sistema de EDOs
def sistema(X, t, alpha, beta):
    """Sistema de duas variaveis acopladas"""
    x, y = X
    dxdt = alpha*x - beta*x*y
    dydt = -y + x*y
    return [dxdt, dydt]

# Parametros
alpha = 1.0
beta = 0.5

# Condicao inicial
X0 = [2.0, 1.0]

# Tempo
t = np.linspace(0, 20, 1000)

# Resolver
sol = odeint(sistema, X0, t, args=(alpha, beta))

# Plotar serie temporal
plt.figure(figsize=(10, 4))
plt.plot(t, sol[:, 0], 'b-', label='x(t)')
plt.plot(t, sol[:, 1], 'r-', label='y(t)')
plt.xlabel('Tempo')
plt.ylabel('Concentracao')
plt.legend()
plt.grid(True)
plt.title('Resolucao Numerica de EDOs Acopladas')
plt.show()

## 3. Modelo Lotka-Volterra (Predador-Presa)

O modelo predador-presa clássico de Lotka-Volterra descreve a dinâmica oscilatória das populações de presas ($x$) e predadores ($y$):
$$\frac{dx}{dt} = \alpha x - \beta xy$$
$$\frac{dy}{dt} = \delta xy - \gamma y$$

Este sistema exibe trajetórias fechadas periódicas e órbitas concêntricas neutras ao redor de seu ponto de equilíbrio estável não-trivial:
$$x^* = \frac{\gamma}{\delta}, \quad y^* = \frac{\alpha}{\beta}$$

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

def lotka_volterra(X, t, alpha, beta, delta, gamma):
    """Modelo predador-presa"""
    x, y = X
    dxdt = alpha*x - beta*x*y
    dydt = delta*x*y - gamma*y
    return [dxdt, dydt]

# Parametros
alpha, beta, delta, gamma = 1.0, 0.5, 0.5, 1.0

# Condicoes iniciais
X0 = [2.0, 1.0]

# Tempo
t = np.linspace(0, 50, 1000)

# Resolver
sol = odeint(lotka_volterra, X0, t, args=(alpha, beta, delta, gamma))

# Plotar retrato de fase
plt.figure(figsize=(8, 6))
plt.plot(sol[:, 0], sol[:, 1], 'b-', linewidth=2, label='Trajetoria')
plt.plot(gamma/delta, alpha/beta, 'ro', markersize=10, label='Equilibrio $(x^*, y^*)$')
plt.xlabel('Presas (x)')
plt.ylabel('Predadores (y)')
plt.legend()
plt.grid(True)
plt.title('Retrato de Fase do Modelo Lotka-Volterra')
plt.show()

## 4. Análise de Estabilidade Linear (Jacobiano e Autovalores)

Para caracterizar a estabilidade local de um equilíbrio de EDOs não-lineares, realizamos a linearização do sistema calculando a matriz Jacobiana simbólica $\vect{J}$ e seus autovalores no ponto fixo.

Dadas as equações:
$$\frac{dx}{dt} = x(1 - x - 0.5y)$$
$$\frac{dy}{dt} = y(-0.75 + 0.5x)$$

Abaixo, usamos a biblioteca simbólica `SymPy` para calcular o Jacobiano analítico e encontrar o ponto fixo. Em seguida, usamos `SciPy` para avaliar numericamente os autovalores de $\vect{J}(1.5, 1.0)$ e determinar a estabilidade da espiral estável.

In [ ]:
import numpy as np
from scipy.linalg import eig
import sympy as sp

# Definir variaveis simbolicas
x, y = sp.symbols('x y')

# Definir sistema
f1 = x*(1 - x - 0.5*y)
f2 = y*(-0.75 + 0.5*x)

# Calcular Jacobiano simbolicamente
J = sp.Matrix([[sp.diff(f1, x), sp.diff(f1, y)],
               [sp.diff(f2, x), sp.diff(f2, y)]])

print("Jacobiano simbolico:")
sp.pprint(J)

# Encontrar pontos de equilibrio
equilibria = sp.solve([f1, f2], [x, y])
print("\nPontos de equilibrio encontrados:", equilibria)

# Avaliar no equilibrio (exemplo: x=1.5, y=1)
x_star, y_star = 1.5, 1.0
J_numeric = np.array(J.subs([(x, x_star), (y, y_star)])).astype(float)

# Calcular autovalores
eigenvalues, eigenvectors = eig(J_numeric)
print(f"\nAutovalores no ponto fixo ({x_star}, {y_star}): {eigenvalues}")
print("Estavel? (Parte real negativa para todos):", all(np.real(eigenvalues) < 0))

## 5. Gerando Diagramas de Bifurcação (Pitchfork Supercrítica)

Uma bifurcação descreve uma mudança qualitativa no comportamento dinâmico de um sistema biológico em resposta à variação contínua de um parâmetro de controle. 
Abaixo, geramos numericamente o diagrama de bifurcação completo para a forma normal da **Bifurcação Pitchfork Supercrítica**:
$$\frac{dx}{dt} = \mu x - x^3$$

Varemos o parâmetro $\mu$ e para cada valor, encontramos seus pontos de equilíbrio usando `fsolve` e analisamos sua estabilidade local a partir da derivada $f'(x^*) = \mu - 3(x^*)^2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

def pitchfork(x, mu):
    """Bifurcacao pitchfork supercritica"""
    return mu*x - x**3

# Varrer parametro mu
mu_vals = np.linspace(-2, 2, 100)
equilibria = []

for mu in mu_vals:
    # Encontrar todos os equilibrios possiveis usando diferentes estimativas iniciais
    eq_list = []
    for x0 in [-2.0, 0.0, 2.0]:  # tentativas iniciais
        eq = fsolve(pitchfork, x0, args=(mu,))[0]
        if abs(pitchfork(eq, mu)) < 1e-6:  # verificar solucao
            eq_list.append(eq)

    # Remover duplicatas numéricas com arredondamento
    eq_list = list(set(np.round(eq_list, 6)))

    for eq in eq_list:
        # Verificar estabilidade linear: derivada de f em x: df/dx = mu - 3*x^2
        stability = mu - 3 * (eq**2)
        if stability < 0:  # estavel
            equilibria.append((mu, eq, 'stable'))
        else:  # instavel
            equilibria.append((mu, eq, 'unstable'))

# Separar os dados para plotagem
stable_mu = [m for m, x, s in equilibria if s == 'stable']
stable_x = [x for m, x, s in equilibria if s == 'stable']
unstable_mu = [m for m, x, s in equilibria if s == 'unstable']
unstable_x = [x for m, x, s in equilibria if s == 'unstable']

# Plotar diagrama
plt.figure(figsize=(8, 5))
plt.scatter(stable_mu, stable_x, color='blue', s=8, label='Estavel')
plt.scatter(unstable_mu, unstable_x, color='red', s=8, facecolors='none', edgecolors='red', label='Instavel')
plt.axvline(0, color='black', linestyle=':', alpha=0.5)
plt.axhline(0, color='gray', linestyle='--', alpha=0.3)
plt.xlabel('Parametro $\mu$')
plt.ylabel('Equilibrio $x^*$')

# Evitar duplicar itens na legenda
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), loc='upper left')

plt.title('Diagrama de Bifurcacao Pitchfork Supercritica')
plt.grid(True)
plt.show()

## 6. Oscilações Não-Lineares e Ciclos Limite (Oscilador de Van der Pol)

O oscilador de Van der Pol descreve um sistema dinâmico não-linear clássico com amortecimento não-linear que exibe um ciclo limite estável robusto:
$$\frac{d^2x}{dt^2} - \mu(1 - x^2)\frac{dx}{dt} + x = 0$$

Convertido em um sistema de EDOs de 1ª ordem:
$$\frac{dx}{dt} = y$$
$$\frac{dy}{dt} = \mu(1 - x^2)y - x$$

Abaixo, simulamos a evolução temporal e construímos o retrato de fase no plano $(x, y)$ mostrando a convergência das trajetórias ao ciclo limite robusto.

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

def van_der_pol(X, t, mu):
    """Oscilador de Van der Pol"""
    x, y = X
    dxdt = y
    dydt = mu*(1 - x**2)*y - x
    return [dxdt, dydt]

# Parametros
mu = 2.0

# Condicao inicial
X0 = [0.1, 0.1]  # Perto da origem (equilibrio instavel)

# Tempo
t = np.linspace(0, 50, 2000)

# Resolver
sol = odeint(van_der_pol, X0, t, args=(mu,))

# Plotar series e retrato de fase
plt.figure(figsize=(12, 5))

# Serie temporal x(t)
plt.subplot(1, 2, 1)
plt.plot(t, sol[:, 0], 'b-', label='x(t)')
plt.xlabel('Tempo')
plt.ylabel('x')
plt.grid(True)
plt.legend()
plt.title('Serie Temporal: Oscilador de Van der Pol')

# Retrato de fase y vs. x
plt.subplot(1, 2, 2)
plt.plot(sol[:, 0], sol[:, 1], 'r-', linewidth=1.0, label='Trajetoria')
plt.plot(sol[0, 0], sol[0, 1], 'go', markersize=8, label='Condicao Inicial')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.title('Retrato de Fase: Convergencia para o Ciclo Limite')

plt.tight_layout()
plt.show()

## 7. Análise de Sensibilidade Paramétrica Local

A análise de sensibilidade local calcula a resposta de um modelo biológico a pequenas perturbações em torno de um conjunto nominal de parâmetros.

Para a reação enzimática em estado estacionário:
$$\frac{dx}{dt} = \frac{V_{max} \cdot S}{K_M + S} - k_{deg} \cdot x = 0 \quad \Rightarrow \quad x^* = \frac{V_{max} \cdot S}{k_{deg}(K_M + S)}$$

Aqui calculamos o coeficiente de sensibilidade local normalizado por diferença finita diante de uma perturbação de $\delta = 1\%$ em cada parâmetro:
$$S_{x^*, p} = \frac{x^*(p + \Delta p) - x^*(p)}{x^*(p)} \cdot \frac{p}{\Delta p}$$

Os valores nominais adotados são: $V_{max} = 10$, $K_M = 5$, $k_{deg} = 0.5$ com substrato fixo $S = 8$.

In [ ]:
import numpy as np
from scipy.integrate import odeint

def modelo_sistema(x, t, Vmax, Km, kdeg, S):
    """Modelo com parametros"""
    dxdt = (Vmax * S) / (Km + S) - kdeg * x
    return dxdt

# Parametros nominais
params_nom = {'Vmax': 10.0, 'Km': 5.0, 'kdeg': 0.5, 'S': 8.0}

# Calcular sensibilidade por perturbacao
def calc_sensibilidade(param_name, delta=0.01):
    """Calcula coeficiente de sensibilidade local"""
    # Valor nominal base
    params_base = params_nom.copy()
    t = np.linspace(0, 50, 500)
    x0 = 0.0
    sol_base = odeint(modelo_sistema, x0, t,
                      args=(params_base['Vmax'], params_base['Km'],
                            params_base['kdeg'], params_base['S']))
    y_base = sol_base[-1, 0]  # Concentracao de estado estacionario

    # Perturbar parametro especifico
    params_pert = params_base.copy()
    params_pert[param_name] *= (1 + delta)
    sol_pert = odeint(modelo_sistema, x0, t,
                      args=(params_pert['Vmax'], params_pert['Km'],
                            params_pert['kdeg'], params_pert['S']))
    y_pert = sol_pert[-1, 0]  # Novo estado estacionario

    # Sensibilidade normalizada
    S_val = ((y_pert - y_base) / y_base) / delta
    return S_val

# Calcular para cada parametro principal
print("Coeficientes de Sensibilidade Local Normalizados:")
for param in ['Vmax', 'Km', 'kdeg']:
    S = calc_sensibilidade(param)
    print(f"  S_(x*, {param}) = {S:.3f}")

## 8. Análise de Sensibilidade Global (GSA) - Método de Sobol

Enquanto a análise local investiga apenas o ponto nominal de operação, a **Análise de Sensibilidade Global (GSA)** estuda como a variabilidade conjunta de todos os parâmetros em intervalos amplos afeta a resposta do sistema. 

Abaixo, aplicamos o **Método de Sobol** através da biblioteca `SALib` para decompor a variância do estado estacionário $x^*$ frente a variações simultâneas de $\pm 50\%$ nos parâmetros $V_{max}$, $K_M$ e $k_{deg}$.

- **Índice de Primeira Ordem ($S1$)**: Fração da variância total da saída devido unicamente ao efeito individual de cada parâmetro.
- **Índice Total ($ST$)**: Fração da variância da saída explicada pelo parâmetro individual mais todos os seus efeitos de interação de ordem superior com outros parâmetros.

In [ ]:
from SALib.sample import saltelli
from SALib.analyze import sobol
import numpy as np

# Definir o problema de variacao dos parametros
problem = {
    'num_vars': 3,
    'names': ['Vmax', 'Km', 'kdeg'],
    'bounds': [[5.0, 15.0],    # Vmax nominal=10 (50% de variacao)
               [2.5, 7.5],     # Km nominal=5
               [0.25, 0.75]]   # kdeg nominal=0.5
}

# Gerar amostras parametricas estruturadas usando a sequencia de Sobol (Saltelli)
n_samples = 1024
param_values = saltelli.sample(problem, n_samples)

# Avaliar o modelo para cada conjunto de amostras parametricas
Y = np.zeros([param_values.shape[0]])

for i, params in enumerate(param_values):
    Vmax, Km, kdeg = params
    # Resolver analiticamente ate o estado estacionario do modelo enzimatico com S=8
    x_ss = (Vmax * 8.0) / (kdeg * (Km + 8.0))
    Y[i] = x_ss

# Executar analise quantitativa de Sobol
Si = sobol.analyze(problem, Y)

print("Indices de Sobol de Primeira Ordem (Efeitos Individuais Principais):")
for name, s1 in zip(problem['names'], Si['S1']):
    print(f"  S1_{name} = {s1:.3f}")

print("\nIndices de Sobol de Efeito Total (Efeito Individual + Interacoes):")
for name, st in zip(problem['names'], Si['ST']):
    print(f"  ST_{name} = {st:.3f}")